# GeoLift market selection — Python walkthrough (`geolift_fast`)

This notebook reproduces Meta GeoLift's **market-selection / power** walkthrough using the
Python port [`geolift_fast`](README.md). It is the *same example* as the R library — read a
panel, pick candidate test markets, run the power simulation across a grid of effect sizes,
and rank the markets by their minimum detectable effect (MDE) — only it runs ~10-14x faster
single-threaded while reproducing R's per-cell decisions to solver tolerance (see
[`../REPORT.md`](../REPORT.md) §7).

### R → Python, step for step

| GeoLift (R) | `geolift_fast` (Python) |
|---|---|
| `GeoDataRead(data, date_id, location_id, Y_id, …)` | `Panel.from_long_csv(path)` / `Panel.from_long_df(df)` |
| `GeoLiftMarketSelection(…)$PowerCurves` | `power_curves(panel, combos, …)` |
| `GeoLiftMarketSelection(…)$BestMarkets` | `best_markets(power_curves_df)` |
| `N = c(2)` (treatment size = 2) | `all_pairs(panel, size=2)` |

The R walkthrough this mirrors:

```r
library(GeoLift)
data(GeoLift_Test)
GeoTestData <- GeoDataRead(data = GeoLift_Test, date_id = "date",
                          location_id = "location", Y_id = "Y",
                          X = c(), format = "yyyy-mm-dd", summary = FALSE)
MarketSelection <- GeoLiftMarketSelection(data = GeoTestData,
                          treatment_periods = c(14), N = c(2),
                          Y_id = "Y", location_id = "location", time_id = "time",
                          effect_size = seq(-0.10, 0.10, 0.05),
                          ns = 1000)
MarketSelection$BestMarkets
```

## 0. Install

From GitHub (works on a GCP VM), or `pip install -e .` from the repo root for local dev:

```bash
pip install "git+https://github.com/moraisnara/geolift-port.git@v0.1.0"
```

Dependencies (`numpy`, `pandas`, `scipy`, `osqp`) resolve automatically. This notebook lives
inside the repo, so it imports the package directly.

In [ ]:
import pandas as pd
from geolift_fast import Panel, power_curves, best_markets, all_pairs

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. Read the panel  —  `GeoDataRead` → `Panel.from_long_csv`

GeoLift's `GeoDataRead` ingests a long/tidy panel (one row per `location`×`date`) and assigns
an integer `time = 1…T` in date order. `Panel.from_long_csv` does the same: it lowercases
locations and orders time by sorted unique date, so the resulting wide outcome matrix matches
R cell-for-cell.

We use the frozen example panel shipped in the repo (`exploration/data/ms_subset_panel.csv` —
24 markets × 220 daily periods), the same data the head-to-head validation in REPORT §7a runs on.

In [ ]:
panel = Panel.from_long_csv("../exploration/data/ms_subset_panel.csv")
print(f"{len(panel.locations)} markets × {panel.T} periods")
panel.locations[:5]

## 2. Candidate test markets  —  `N = c(2)` → `all_pairs(panel, size=2)`

In R, `N = c(2)` tells `GeoLiftMarketSelection` to evaluate every treatment group of size 2.
`all_pairs(panel, size=2)` builds exactly that list of candidate combos (you can also pass your
own list of lists, e.g. `[["market_001", "market_017"], …]`, or restrict the pool with the
`locations=` argument).

In [ ]:
combos = all_pairs(panel, size=2)
print(f"{len(combos)} candidate market pairs")
combos[:3]

## 3. Power simulation  —  `GeoLiftMarketSelection(…)$PowerCurves` → `power_curves(…)`

`power_curves` runs GeoLift's power simulation for every (combo × test-window × effect size)
cell: it fits the fixed-effects synthetic control on the pre-period, computes the ATT and
scaled-L2 imbalance, and gets the conformal “iid” permutation p-value. It returns the same tidy
table as R's `$PowerCurves`.

Same knobs as the R call above: `treatment_periods=14`, `effect_sizes=seq(-0.10, 0.10, 0.05)`,
`ns=1000` (Meta default), `seed=42` (statistics are deterministic given the seed).

In [ ]:
pc = power_curves(
    panel, combos,
    treatment_periods=14,                              # int or list of test-window lengths
    effect_sizes=[-0.10, -0.05, 0.0, 0.05, 0.10],
    ns=1000,                                            # conformal permutations (Meta default)
    alpha=0.10,
    seed=42,
)
# columns: location, duration, EffectSize, pvalue, power, AvgATT, AvgScaledL2Imbalance
pc.head(10)

## 4. Rank the markets  —  `GeoLiftMarketSelection(…)$BestMarkets` → `best_markets(…)`

`best_markets` ranks each candidate by its **minimum detectable effect (MDE)** — the smallest
positive effect size that comes out significant at `alpha` — best (smallest MDE) first. This is
the analogue of GeoLift's `$BestMarkets`: the markets at the top are the most sensitive test
designs (they can detect smaller lifts).

In [ ]:
ranking = best_markets(pc, alpha=0.10)   # columns: rank, location, duration, MDE
ranking.head(10)

## 5. Inspect a chosen market

Slice the power curve for the top-ranked pair to see how power rises with effect size — the
same diagnostic you'd read off `MarketSelection$PowerCurves` in R.

In [ ]:
top = ranking.loc[0, "location"]
pc[pc.location == top][["EffectSize", "pvalue", "power", "AvgATT", "AvgScaledL2Imbalance"]]

## Lower-level building blocks (optional)

For custom pipelines, the per-combo engine is exposed directly. `simulate_combo` fits one combo
and returns its cells across all effect sizes (reusing the effect-size-invariant pre-period fit
— GeoLift's “lever 2”). `ComboFit`, `scm_weights`, `conformal_resids`, and `conformal_pval` give
you the SCM weights, residuals, and permutation p-value individually.

```python
import numpy as np
from geolift_fast import simulate_combo

rng = np.random.default_rng(42)
cells = simulate_combo(panel, ["market_001", "market_017"], tp=14,
                       effect_sizes=[0.0, 0.05, 0.10], ns=1000, rng=rng, alpha=0.10)
```

### Fidelity
Validated against the R engine on this exact panel: per-cell significance and selected-market
ranking **identical**, ATT to ~1e-7, scaled-L2 to ~1e-8; p-values differ only by Monte-Carlo
error (independent RNG). See [`../REPORT.md`](../REPORT.md) §7a for the head-to-head.